In [ ]:
%%capture
%cd ~/repos/PredUCE/
%load_ext autoreload
%autoreload 2

In [ ]:
import polars as pl

from make_clinical_dataset.epic.combine import merge_closest_measurements
from make_clinical_dataset.shared.constants import ROOT_DIR
from preduce.emerg.config import EMBEDDING_SECTIONS

EMB_MODEL = "PubMedBERT"
DATE = '2025-03-29'
DATA_DIR = f"{ROOT_DIR}/data/final/data_{DATE}"
DATA_PATH = f'{DATA_DIR}/processed/treatment_centered_data.parquet'
DATE_PATH = f'{DATA_DIR}/processed/treatment_centered_dates.parquet'
NOTE_PATH = f'{DATA_DIR}/interim/subsets/clinic_visits_prior_to_treatment/notes.parquet'

TEXT_PATH = f'{DATA_DIR}/interim/embedding/ed_risk_summary.parquet'
EMB_PATH = f'{DATA_DIR}/interim/embedding/{EMB_MODEL}'

In [ ]:
# Load data
tabular_df = pl.read_parquet(DATA_PATH)
note_df = pl.read_parquet(NOTE_PATH, columns=["mrn", "clinic_date", "note_id"])
text_df = pl.read_parquet(TEXT_PATH, columns=["note_id"] + [f"{section}_text_id" for section in EMBEDDING_SECTIONS])

# Get the closest clinical note prior to assessment date within the lookback window
note_df = note_df.rename({"clinic_date": "prev_clinic_date", "note_id": "prev_note_id"})
df = merge_closest_measurements(tabular_df, note_df, "assessment_date", "prev_clinic_date", merge_individually=False, time_window=(-30,-1))
del tabular_df, note_df

# Include the text section ids
df = df.join(text_df, left_on="prev_note_id", right_on="note_id", how="left")

In [ ]:
# TODO: make it robust to missing columns 
# keep only the first treatment of a given week
df = df.group_by_dynamic("assessment_date", every="7d", group_by="mrn").agg(pl.all().first())

# create an indicator on whether this is patient's very first treatment
# TODO: move this to make-clinical-dataset
df = df.with_columns(
    pl.col("days_since_last_treatment").is_null().alias("no_prior_treatment"),
)

# one-hot-encode categorical columns with low-cardinality
# WARNING: assumes categories will remain constant over time
df = df.to_dummies(columns=DEFAULT_ENCODE_COLS)

# clip columns hueristically before imputation
df = df.with_columns([
    pl.col("days_since_starting_treatment").clip(lower_bound=-1),
    pl.col("prev_hospitalization_length_of_stay").clip(lower_bound=1),
])

# impute columns heuristrically (i.e. fill with zero)
df = df.with_columns([
    *[pl.col(col).fill_null(0) for col in CANCER_DRUGS],
    pl.col("radiation_dose_given").fill_null(0),
    pl.col("prev_hospitalization_length_of_stay").fill_null(0)
])

# impute columns via missing indicator approach (MIA)
# use only the columns that exist in the data
cols = [col for col in DEFAULT_IMPUTE_COLS if col in df.columns]
df = df.with_columns([
    # create missingness indicators for select columns
    *[pl.col(col).is_null().alias(f'{col}_missing') for col in cols],

    # fill missing values with -1
    # NOTE: the model will learn from the appropriate indicator to ignore this value
    #   i.e. ignore  "days_since_prev_ED_visit" when "num_prior_ED_visits_within_5_years" == 0
    #   i.e. ignore  "days_since_last_treatment" when "no_prior_treatment" == 1
    #   i.e. ignore "hemoglobin" when "hemoglobin_missing" == 1
    *[pl.col(col).fill_null(-1) for col in cols],
])
# drop unnecessary missingness indicators
df = df.drop(
    "days_since_last_treatment_missing",
    "days_since_prev_ED_visit_missing",
    "days_since_prev_hospitalization_missing",
    strict=False

# split the data into train, valid, test set
split_date = "2022-01-01"
splitter = Splitter()
train_data, valid_data, test_data = splitter.split_data(
    df, visit_col="assessment_date", split_date=split_date
)

# transform the three datasets BASED ON the train set
# remove uninformative columns (low variance or highly correlated)
# clip outliers
# normalize data
# combine the datasets back for convenience
# split data into input tabular features, input embedding features, output targets, meta info
)

In [ ]:
meta_cols = [
    "mrn",
    "assessment_date",
    "split",
    "cancer_type",
    "cancer_desc",
    "morphology_desc",
    "primary_site_code",
    "preferred_language",
    "religion",
    "postalcode",
    "department",
    "regimen",
    "prev_hospitalization_note",
    "prev_ED_visit_note",
    "prev_ED_visit_CTAS_score",
]